In [1]:
import re
import string


import pandas as pd

import spacy
import en_core_web_sm

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
# Carga de datos de train
df_train = pd.read_json("data/train.jsonl",lines=True)

In [3]:
# Métodos de preprocesado de texto

translator = str.maketrans('', '', string.punctuation)
nlp = en_core_web_sm.load()

def limpieza_comun(texto: str) -> str:
    # El dataset contiene '\' que deben ser eliminadas manualmente
    texto = re.sub(r'\\', ' ', texto)
    # Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def preprocessing_minimal(texto: str) -> str:
    """
    Mantiene puntuación, mayúsculas y estructura.
    Útil para sistemas más complejos como LLMs y Transformers.
    """
    return limpieza_comun(texto)

def preprocessing_basico(texto: str) -> str:
    """
    Limpia el texto convirtiéndolo todo a minúsculas y eliminando signos de puntuación.
    Útil para TF-IDF o Word2Vec
    """ 
    texto = limpieza_comun(texto)
    return texto.lower().translate(str.maketrans('', '', string.punctuation))

def preprocessing_avanzado(texto: str) -> str:
    """
    Limpia el texto convirtiéndolo todo a minúsculas y eliminando signos de puntuación. 
    Útil para modelos estadísticos básados en frecuencias.
    """
    texto = limpieza_comun(texto).lower()
    doc = nlp(texto)
    # Añadimos filtro de .is_alpha para quitar números o símbolos raros que queden
    textoLimpio = [word.lemma_ for word in doc if (not (word.is_stop or word.is_punct or not word.is_alpha)and word.is_alpha)]
    return " ".join(textoLimpio)


example = df_train.description[0]

print(f"Ejemplo: {example}")
print("--------"*15)
print(f"1. Preprocesado MINIMAL: {preprocessing_minimal(example)}")
print(f"2. Preprocesado BASIC: {preprocessing_basico(example)}")
print(f"3. Preprocesado ADVANCE: {preprocessing_avanzado(example)}")
print("--------"*15)

Ejemplo: Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
------------------------------------------------------------------------------------------------------------------------
1. Preprocesado MINIMAL: Reuters - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.
2. Preprocesado BASIC: reuters  shortsellers wall streets dwindling band of ultracynics are seeing green again
3. Preprocesado ADVANCE: reuters short seller wall street dwindle band ultra cynic see green
------------------------------------------------------------------------------------------------------------------------


In [ ]:
# Vectorización Dispersa vs Densa

# Dispersas

def BoW(textos: pd.Series) -> pd.DataFrame:
    """
    Convierte todos los textos en un Bag Of Words
    limitado a las palabras más frecuentes de los textos.
    """
    # max_features limita a las N palabras más frecuentes
    vectorizador = CountVectorizer(max_features=5000, min_df=2, stop_words='english')
    
    X = vectorizador.fit_transform(textos)
    
    return pd.DataFrame(X.toarray(), columns=vectorizador.get_feature_names_out())

def TF_IDF(textos: pd.Series) -> pd.DataFrame:
    """
    Convierte todos los textos en una matriz con TF-IDF.
    """
    vectorizador = TfidfVectorizer(max_features=5000, min_df=2, stop_words='english')
    
    X = vectorizador.fit_transform(textos)
    
    return pd.DataFrame(X.toarray(), columns=vectorizador.get_feature_names_out())


df_bow = BoW(df_train.description.apply(preprocessing_avanzado))
df_tfidf = TF_IDF(df_train.description.apply(preprocessing_basico))